# **EDA - Predikcija cene polovnih automobila**

U ovoj svesci istraženi su podaci koji su osnova za kreiranje, treniranje i testiranje modela predikcije cene polovnih automobila.

Cilj je da se na kraju analize sačini spisak radnih zadataka u svrhu čišćenja podataka radi dalje upotrebe.

## KORAK 1: Učitavanje podataka

In [29]:
from pathlib import Path
import pandas as pd

In [30]:
DATA_PATH = Path("../data/cars.csv")
df = pd.read_csv(DATA_PATH)

## KORAK 2: Prikaz podataka

Prikaz prvih nekoliko redova:

In [31]:
df.head()

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
0,mazda,2,5500,2008,with mileage,162000.0,petrol,1500.0,burgundy,mechanics,front-wheel drive,B
1,mazda,2,5350,2009,with mileage,120000.0,petrol,1300.0,black,mechanics,front-wheel drive,B
2,mazda,2,7000,2009,with mileage,61000.0,petrol,1500.0,silver,auto,front-wheel drive,B
3,mazda,2,3300,2003,with mileage,265000.0,diesel,1400.0,white,mechanics,front-wheel drive,B
4,mazda,2,5200,2008,with mileage,97183.0,diesel,1400.0,gray,mechanics,front-wheel drive,B


Prikaz nasumično odabranih 5 redova:

In [32]:
df.sample(5, random_state=42)

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
19270,mitsubishi,carisma,2050,1999,with mileage,250000.0,petrol,1800.0,blue,mechanics,front-wheel drive,M
25927,ford,fusion,4500,2006,with mileage,160000.0,petrol,1400.0,burgundy,mechanics,front-wheel drive,M
23388,mercedes-benz,e-klass,3500,1999,with mileage,485000.0,diesel,2900.0,blue,mechanics,rear drive,E
53189,bmw,x3,21000,2013,with mileage,159000.0,diesel,2000.0,black,auto,NaN,J
34058,renault,megane,3990,2002,with mileage,331700.0,diesel,1900.0,other,mechanics,front-wheel drive,C


Neke nazive kolona (na primer, mileage(kilometers)) je potrebno korigovati tako da budu snake_case.

Iz slučajno odabranih redova za prikaz vidi se da u nekim kolonama ima i nedostajućih vrednosti.

## KORAK 3: Osnovna struktura skupa podataka

In [33]:
df.shape
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 56244
Columns: 12


Imamo 56244 redova, odnosno unosa o prodaji automobila.

In [34]:
df.columns.to_list()

['make',
 'model',
 'priceUSD',
 'year',
 'condition',
 'mileage(kilometers)',
 'fuel_type',
 'volume(cm3)',
 'color',
 'transmission',
 'drive_unit',
 'segment']

Nazive kolona treba korigovati tako sva slova budu mala i da nazivi kolona, sastavljeni od više reči, budu u tzv.snake_case prikazu.

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 56244 entries, 0 to 56243
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   make                 56244 non-null  str    
 1   model                56244 non-null  str    
 2   priceUSD             56244 non-null  int64  
 3   year                 56244 non-null  int64  
 4   condition            56244 non-null  str    
 5   mileage(kilometers)  56244 non-null  float64
 6   fuel_type            56244 non-null  str    
 7   volume(cm3)          56197 non-null  float64
 8   color                56244 non-null  str    
 9   transmission         56244 non-null  str    
 10  drive_unit           54339 non-null  str    
 11  segment              50953 non-null  str    
dtypes: float64(2), int64(2), str(8)
memory usage: 5.1 MB


Nedostajućih vrednosti ima u kolonama: volume(cm3), drive_unit i segment. Bitno je da nedostajućih vrednosti nema u ciljnoj promenljivoj - priceUSD.

Tipovi podataka u kolonama su u redu, logično dodeljeni.

## KORAK 4: Analiza ciljne promenljive

**Ciljna promenljiva** u ovom projektu je **priceUSD**. Bacamo pogled na vrednosti u ovoj koloni i proveravamo tip podataka.

In [36]:
df["priceUSD"].head()

0    5500
1    5350
2    7000
3    3300
4    5200
Name: priceUSD, dtype: int64

In [37]:
df["priceUSD"].dtype

dtype('int64')

In [38]:
df["priceUSD"].unique()

array([5500, 5350, 7000, ..., 5012, 3922, 4492], shape=(2970,))

In [39]:
df["priceUSD"].describe()

count     56244.000000
mean       7415.456440
std        8316.959261
min          48.000000
25%        2350.000000
50%        5350.000000
75%        9807.500000
max      235235.000000
Name: priceUSD, dtype: float64

Iz deskriptivne statistike ciljne promenljive vidimo da je minimalna cena automobila 48 dolara. To je sumnjivo i verovatno se radi o grešci.

In [ ]:
df.sort_values(by="priceUSD").head(50)

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
5952,zaz,968,48,1910,with damage,25000.0,petrol,1100.0,green,mechanics,rear drive,B
40252,proton,persona,95,1997,with damage,1111.0,petrol,1600.0,burgundy,mechanics,front-wheel drive,NaN
24873,ford,fiesta,100,1992,for parts,1.0,petrol,1100.0,blue,mechanics,front-wheel drive,B
5456,mazda,626,100,1986,with mileage,100.0,petrol,2000.0,gray,mechanics,front-wheel drive,C
54171,citroen,xantia,100,1995,for parts,380000.0,petrol,1800.0,silver,mechanics,front-wheel drive,D
20971,toyota,corolla,100,1988,for parts,300000.0,petrol,1300.0,red,mechanics,front-wheel drive,C
41010,nissan,primera,100,1994,for parts,300000.0,petrol,2000.0,burgundy,mechanics,front-wheel drive,D
23898,ford,escort,100,1986,for parts,200000.0,petrol,1300.0,blue,mechanics,front-wheel drive,C
7314,mercedes-benz,190-w201,100,1991,for parts,2000.0,petrol,2000.0,black,mechanics,NaN,D
7076,lada-vaz,21099,119,1992,for parts,20000.0,petrol,1500.0,green,mechanics,front-wheel drive,B


In [52]:
df.sort_values(by="priceUSD").tail(50)

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
50780,toyota,tundra,90000,2019,with mileage,100.0,petrol,5700.0,black,auto,part-time four-wheel drive,NaN
42491,land-rover,range-rover,91000,2017,with mileage,45000.0,petrol,5000.0,white,auto,all-wheel drive,J
27121,mercedes-benz,gls-amg,92000,2016,with mileage,41000.0,petrol,5500.0,gray,auto,all-wheel drive,J
41988,audi,q8,92369,2019,with mileage,10.0,petrol,2995.0,blue,auto,all-wheel drive,NaN
34505,tesla,model-x,92500,2017,with mileage,8000.0,electrocar,NaN,blue,auto,all-wheel drive,NaN
27112,mercedes-benz,gls,92717,2018,with mileage,11.0,diesel,3000.0,blue,auto,part-time four-wheel drive,J
34509,tesla,model-x,93000,2017,with mileage,14000.0,electrocar,NaN,red,auto,all-wheel drive,NaN
37780,porsche,panamera,93000,2017,with mileage,20000.0,petrol,3000.0,blue,auto,all-wheel drive,S
20197,mercedes-benz,cl-amg,93595,2004,with mileage,69652.0,petrol,5980.0,black,auto,rear drive,F
22093,aston-martin,dbs,95000,2010,with mileage,35000.0,petrol,5900.0,gray,auto,rear drive,NaN


Postoji problem sa sumnjivo niskim cenama, dok najviše cene u skupu podataka su moguće za vrlo skupe modele.

## KORAK 5: Provera nedostajućih vrednosti 

In [40]:
missing_values = df.isna().sum()
missing_values[missing_values > 0]

volume(cm3)      47
drive_unit     1905
segment        5291
dtype: int64

U tri kolone imamo nedostajuće vrednosti. Te redove treba ili izbrisati ili popuniti prosečnom vrednošću, medijanom ili najčešćom vrednošću. 
Kolonu *segment* sigurno treba popuniti, jer bi se brisanjem redova koji ovde sadrže nedostajuće vrednosti izgubio veliki deo skupa, oko 10%.

In [41]:
missing_like_values = ["NaN", "nan", "NULL", "null", "None", "none", "", " "]
 
for column in df.columns:
    if df[column].dtype == "object":
        count = df[column].astype(str).str.strip().isin(missing_like_values).sum()
         
        if count > 0:
            print(f"{column}: {count}")

U kolonama ovog dataframe-a nema vrednosti koje izgledaju kao tekst, ali zapravo predstavljaju nedostajuće podatke.

## KORAK 6: Analiza numeričkih kolona 

U ovom skupu podataka numeričke kolone mogu biti: 'priceUSD', 'year', 'mileage(kilometers)', 'volume(cm3)'.

In [42]:
numeric_columns = ['priceUSD', 'year', 'mileage(kilometers)', 'volume(cm3)']
df[numeric_columns].dtypes

priceUSD                 int64
year                     int64
mileage(kilometers)    float64
volume(cm3)            float64
dtype: object

Sve numeričke kolone su odgovarajućeg tipa podataka.

In [43]:
df[numeric_columns].describe()

,priceUSD,year,mileage(kilometers),volume(cm3)
count,56244.000000,56244.000000,5.624400e+04,56197.000000
mean,7415.456440,2003.454840,2.443956e+05,2104.860615
std,8316.959261,8.144247,3.210307e+05,959.201633
min,48.000000,1910.000000,0.000000e+00,500.000000
25%,2350.000000,1998.000000,1.370000e+05,1600.000000
50%,5350.000000,2004.000000,2.285000e+05,1996.000000
75%,9807.500000,2010.000000,3.100000e+05,2300.000000
max,235235.000000,2019.000000,9.999999e+06,20000.000000


Pored minimalne vrednosti u koloni priceUSD, sumnjiva je maksimalna vrednost kilometraže u koloni mileage(kilometers). Sve kilometraže preko 1.000.000 km su vrlo retke. Vrednosti preko 1.000.000 treba zameniti tom vrednošću - 1.000.000 km. 

Sumnjiva je i maksimalna vrednost u koloni volume(cm3), s obzirom da kubikaža putničkih automobila ne prelazi 8.000 cm3. Sve vrednosti preko 8.000 cm3 treba zameniti vrednošću od 8.000 cm3. Takođe, ni minimalna kubikaža motora ne treba da bude manja od 650 cm3. 

## KORAK 7: Analiza kategorijskih kolona 

U ovom skupu podataka kategorijske kolone mogu biti: 'make', 'model', 'condition', 'fuel_type', 'color', 'transmission', 'drive_unit', 'segment'.

In [44]:
categorical_columns = ['make', 'model', 'condition', 'fuel_type', 'color', 'transmission', 'drive_unit', 'segment']
df[categorical_columns].dtypes

make            str
model           str
condition       str
fuel_type       str
color           str
transmission    str
drive_unit      str
segment         str
dtype: object

In [45]:
for column in categorical_columns:
    print(column)
    print(df[column].nunique())
    print("-" * 40)

make
96
----------------------------------------
model
1034
----------------------------------------
condition
3
----------------------------------------
fuel_type
3
----------------------------------------
color
13
----------------------------------------
transmission
2
----------------------------------------
drive_unit
4
----------------------------------------
segment
9
----------------------------------------


In [46]:
for column in categorical_columns:
    print(df[column].value_counts(dropna=False))
    print("-" * 40)

make
volkswagen    6861
audi          4030
bmw           4013
opel          3779
renault       3713
              ... 
trabant          1
jac              1
asia             1
tagaz            1
saipa            1
Name: count, Length: 96, dtype: int64
----------------------------------------
model
passat      2086
5-seriya    1476
a6          1276
golf        1070
astra       1013
            ... 
xb             1
xc40           1
xjs            1
xt5            1
z3             1
Name: count, Length: 1034, dtype: int64
----------------------------------------
condition
with mileage    55278
with damage       512
for parts         454
Name: count, dtype: int64
----------------------------------------
fuel_type
petrol        36405
diesel        19792
electrocar       47
Name: count, dtype: int64
----------------------------------------
color
black       12385
silver      10075
blue         8083
gray         5807
white        5292
green        3911
other        3397
red          2744
bur

U katogorijskim kolonama nisu uočeni problemi.

## KORAK 8: Ekstremne vrednosti 

Ekstremne vrednosti su već razmotrene kroz komentare u prethodnim koracima.

Ovde ćemo proveriti da li postoje nevalidne godine proizvodnje. Serijska proizvodnja automobila počela je 1910. godine. Bolji poznavaoci automobilizma od mene znali bi da li je logično da dole prikazani modeli budu proizvedene u godini koja je upisana u podacima. 

In [53]:
df['year'].describe()

count    56244.000000
mean      2003.454840
std          8.144247
min       1910.000000
25%       1998.000000
50%       2004.000000
75%       2010.000000
max       2019.000000
Name: year, dtype: float64

In [54]:
df.sort_values(by="year").head(50)

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
5952,zaz,968,48,1910,with damage,25000.0,petrol,1100.0,green,mechanics,rear drive,B
43024,eksklyuziv,retro,50000,1933,with mileage,1.0,petrol,2400.0,black,mechanics,NaN,NaN
43020,eksklyuziv,retro,4991,1936,with damage,175000.0,petrol,1800.0,green,mechanics,rear drive,NaN
43025,eksklyuziv,retro,12500,1938,with mileage,100000.0,petrol,3000.0,black,mechanics,rear drive,NaN
42986,eksklyuziv,redkaya-model,17000,1942,with mileage,40000.0,petrol,2000.0,black,mechanics,front-wheel drive,NaN
52637,eksklyuziv,voennaya-tehnika,20000,1945,with mileage,1.0,petrol,2199.0,green,mechanics,part-time four-wheel drive,NaN
1048,gaz,67,9200,1948,with mileage,100000.0,petrol,3200.0,green,mechanics,part-time four-wheel drive,NaN
33000,gaz,m-20-pobeda,2800,1949,with mileage,55555.0,petrol,2400.0,brown,mechanics,NaN,NaN
32995,gaz,m-20-pobeda,2000,1950,with mileage,100000.0,petrol,2400.0,blue,mechanics,NaN,NaN
32998,gaz,m-20-pobeda,5900,1950,with mileage,1111.0,petrol,2200.0,brown,mechanics,rear drive,NaN


## **Spisak problema za fazu čišćenja**

Tokom čišćenja i pripreme podataka treba rešiti sledeće probleme:

- nazive kolona treba standardizovati (snake-case);
- ciljnu kolonu *priceUSD* treba proveriti u pogledu neuobičajeno malih vrednosti;
- kolonu *mileage(kilometers)* treba proveriti u pogledu neuobičajeno velikih vrednosti (preko 1.000.000 km);
- kolonu *volume(cm3)* treba proveriti u pogledu neuobičajeno malih (ispod 650 cm3) i neuobičajeno velikih vrednosti (iznad 8.000 cm3).